<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_07-3_Bounded_Evaluator_Optimizer_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 3. Bounded Evaluator–Optimizer

**학습 목표:** `Generate → Evaluate → Improve`를 반복하되 개발자가 `max_rounds`로 상한을 정하는 **Workflow**를 구현한다.

- **기본 모드:** API 키 없이 실행되는 연습용 LLM 시뮬레이터
- **선택 모드:** Google Gemini API로 실제 초안 생성과 수정
- **평가:** 모델의 주관적 점수 대신, 요구된 항목이 포함되는지 Python 코드로 검사
- **핵심 질문:** 평가 결과에 따라 분기하더라도 왜 Agent가 아닌가?

> 실습용 사례: 지능시스템 과목의 평가 비율을 안내하는 공지문 생성. 예시 평가 비율은 가상의 교육용 데이터입니다.

## 1. 준비: 설정과 평가 기준

In [19]:
# 실습 모드: False면 API 없이 모든 셀 실행 가능
USE_GEMINI = False
MAX_ROUNDS = 3

# 실제 모델명은 사용 가능한 모델 목록과 API 계정 상태를 확인 후 변경할 수 있습니다.
MODEL_NAME = "gemini-3.5-flash-lite"

# 공지문에 반드시 포함해야 할 항목 (가상의 교육용 수치)
REQUIREMENTS = {
    "과제": "20%",
    "중간": "30%",
    "기말": "30%",
    "출석": "20%",
}

print("반복 상한:", MAX_ROUNDS)
print("평가 항목:", REQUIREMENTS)

반복 상한: 3
평가 항목: {'과제': '20%', '중간': '30%', '기말': '30%', '출석': '20%'}


## 2. (선택) 실제 Gemini API 연결

`USE_GEMINI=True`인 경우에만 실행됩니다. Colab의 **🔑 보안 비밀(Secrets)**에 `GEMINI_API_KEY`를 저장하거나 실행 시 입력하세요. 키를 노트북 코드에 직접 입력하지 않습니다. 실제 API 사용량과 무료 한도는 계정/모델별로 다릅니다.

In [20]:
client = None
if USE_GEMINI:
    %pip -q install google-genai
    import os
    from getpass import getpass
    from google import genai

    # Colab Secrets에 저장된 키를 먼저 찾습니다.
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except (ImportError, Exception):
        api_key = None

    if not api_key:
        api_key = getpass("GEMINI_API_KEY를 입력하세요: ")
    if not api_key:
        raise ValueError("API 키가 없어 실제 Gemini 모드를 사용할 수 없습니다.")
    client = genai.Client(api_key=api_key)
    print("Gemini 클라이언트 준비 완료")
else:
    print("모의 LLM 모드: API 키 없이 실습합니다.")

모의 LLM 모드: API 키 없이 실습합니다.


## 3. Generate: 첫 초안 만들기

In [21]:
def call_llm(prompt):
    """실제 Gemini 모드에서만 사용되는 텍스트 생성 함수"""
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    text = response.text or ""
    if not text.strip():
        raise RuntimeError("모델 응답이 비어 있습니다.")
    return text.strip()

def generate():
    """첫 공지문을 생성합니다. 일부 항목을 빠뜨려 개선의 필요성을 보여줍니다."""
    if USE_GEMINI:
        return call_llm(
            "지능시스템 강의 평가 공지문을 한국어로 간결하게 작성하세요. "
            "평가 항목은 과제 20%, 중간 30%, 기말 30%, 출석 20%입니다."
        )
    # 고의로 불완전한 시작 문장: 평가를 거쳐 개선하도록 설계
    return "지능시스템 평가 안내: 과제 20%, 중간 30%입니다."

draft = generate()
print("[초안]", draft)

[초안] 지능시스템 평가 안내: 과제 20%, 중간 30%입니다.


## 4. Evaluate: 명시적인 규칙으로 평가

각 항목의 이름과 비율이 같은 공지문에 포함됐는지 검사합니다. **이 평가는 교육용 단순 검사**로, 의미적 정확성 전체를 보증하지는 않습니다.

In [22]:
def evaluate(draft):
    """모든 필수 항목이 있는지 검사하고 구체적인 피드백을 반환합니다."""
    missing = []
    for item, ratio in REQUIREMENTS.items():
        if item not in draft or ratio not in draft:
            missing.append(f"{item} {ratio}")
    passed = (len(missing) == 0)
    feedback = "모든 필수 항목 포함" if passed else "누락: " + ", ".join(missing)
    return passed, feedback

passed, feedback = evaluate(draft)
print("통과:", passed)
print("피드백:", feedback)

통과: False
피드백: 누락: 기말 30%, 출석 20%


## 5. Improve: 평가 피드백으로 수정

In [23]:
def improve(draft, feedback):
    """현재 초안과 피드백을 받아 수정본을 만듭니다."""
    if USE_GEMINI:
        return call_llm(
            f"다음 공지문을 보완하세요. 기존 내용은 유지하고 누락을 채우세요.\n"
            f"현재 초안: {draft}\n평가 피드백: {feedback}\n"
            f"필수 사항: {REQUIREMENTS}\n간결한 한국어 공지문만 출력하세요."
        )
    # 모의 모드에서는 빠진 항목을 순서대로 보충
    additions = []
    for item, ratio in REQUIREMENTS.items():
        if item not in draft or ratio not in draft:
            additions.append(f"{item} {ratio}")
    if not additions:
        return draft
    return draft.rstrip(". ") + ", " + ", ".join(additions) + "입니다."

print("[개선 예시]", improve(draft, feedback))

[개선 예시] 지능시스템 평가 안내: 과제 20%, 중간 30%입니다, 기말 30%, 출석 20%입니다.


## 6. Bounded Evaluator–Optimizer: 최대 반복 횟수를 코드로 제어

원본 슬라이드와 마찬가지로 `evaluate()` 결과가 동적이어도 **반복 구조와 최대 횟수는 개발자가 고정**합니다. 마지막 회차에서는 불필요하게 `improve()`를 호출하지 않습니다.

In [24]:
def evaluator_optimizer(draft, evaluate, improve, max_rounds=3):
    """평가-수정을 최대 max_rounds회까지 반복하는 고정 Workflow"""
    if max_rounds < 1:
        raise ValueError("max_rounds는 1 이상이어야 합니다.")

    log = []  # 매 회차의 평가 결과 기록
    for round_id in range(1, max_rounds + 1):
        passed, feedback = evaluate(draft)
        log.append({
            "round": round_id,
            "passed": passed,
            "feedback": feedback,
            "draft": draft,
        })

        print(f"[{round_id}회차] 통과={passed} / {feedback}")
        if passed:
            return draft, log, "passed"  # 성공하면 조기 종료

        # 마지막 회차라면 추가 API 호출 없이 즉시 종료
        if round_id < max_rounds:
            draft = improve(draft, feedback)

    return draft, log, "max_rounds_reached"  # 실패한 채 종료할 수도 있음

final_text, history, stop_reason = evaluator_optimizer(
    generate(), evaluate, improve, max_rounds=MAX_ROUNDS
)
print("\n[최종 결과]", final_text)
print("[종료 이유]", stop_reason)

[1회차] 통과=False / 누락: 기말 30%, 출석 20%
[2회차] 통과=True / 모든 필수 항목 포함

[최종 결과] 지능시스템 평가 안내: 과제 20%, 중간 30%입니다, 기말 30%, 출석 20%입니다.
[종료 이유] passed


## 7. 실행 로그 관찰 및 자동 검증

In [25]:
# 최종 결과가 통과했는지, 몇 번 평가했는지 확인
print("평가 횟수:", len(history))
for row in history:
    print(f"round={row['round']} | passed={row['passed']} | {row['feedback']}")

assert len(history) <= MAX_ROUNDS
assert stop_reason in ("passed", "max_rounds_reached")
assert history[-1]["passed"] == (stop_reason == "passed")
if not USE_GEMINI:
    assert stop_reason == "passed"
print("검증 통과: 반복 상한과 종료 조건 정상 동작")

평가 횟수: 2
round=1 | passed=False | 누락: 기말 30%, 출석 20%
round=2 | passed=True | 모든 필수 항목 포함
검증 통과: 반복 상한과 종료 조건 정상 동작


## 8. 실패해도 반드시 종료하는가? (Hard Stop 확인)

In [26]:
def always_fail(_draft):
    return False, "아직 부족함"

def no_change(draft, feedback):
    return draft  # 개선이 전혀 없다고 가정

failed_text, failed_log, reason = evaluator_optimizer(
    "수정되지 않는 초안", always_fail, no_change, max_rounds=3
)
assert len(failed_log) == 3
assert reason == "max_rounds_reached"
print("실패하더라도 3회 후 안전하게 종료:", reason)

[1회차] 통과=False / 아직 부족함
[2회차] 통과=False / 아직 부족함
[3회차] 통과=False / 아직 부족함
실패하더라도 3회 후 안전하게 종료: max_rounds_reached


## 9. 학생 과제 및 토론

1. `MAX_ROUNDS=1`과 `MAX_ROUNDS=3`을 비교하고 결과와 API 호출 수의 차이를 설명하세요.
2. `REQUIREMENTS`에 **팀 프로젝트 10%** 항목을 추가해 보고, 기존 비율 합이 100%가 되도록 다른 비율도 조정하세요.
3. `evaluate()`가 단순 단어 포함이 아니라 `과제 20%`처럼 **항목과 수치가 연속해 나타나는지** 검사하게 개선해 보세요.
4. `max_rounds_reached`일 때 성공이라고 표시하면 왜 위험할까요?

**정리:** `generate → evaluate → improve`라는 실행 경로, 반복 상한, 종료 지점이 코드에 미리 정해져 있으므로 LLM이 초안을 생성하거나 수정해도 이 예제는 **Workflow**입니다.

**참고:** [Google Gen AI SDK 공식 문서](https://googleapis.github.io/python-genai/) · [Google AI Studio API 키](https://aistudio.google.com/apikey)